# 3. Use and fine-tune the MACEField-MH-1 foundation model

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mdi-group/mace-field-tutorial/blob/main/tutorials/macefield/03_foundation_finetuning.ipynb)

This notebook downloads `MACEField-MH-1.model` from the [MACEField 1.1.0 release](https://github.com/mdi-group/mace-field/releases/tag/1.1.0). The checkpoint starts from upstream MACE pretrained model [`mace-mh-1.model`](https://github.com/ACEsuit/mace-foundations/tree/main/mace_mh_1), using its `omat_pbe` head. Its heads are `pt_head`, `MP-ferroelectric`, `MP-Dielectric`, and `Pheasy-MP-Phonon-v1.1`. The `pt_head` single-point examples analyze Ti-displacement polarization and BECs in BTO, then electronic and ionic dielectric response, BECs, and a Hessian in α-quartz SiO₂. An optional fine-tuning example follows.

Fine-tuning is opt-in because it needs a GPU and can take substantial time. The recipe initializes a new `ft_head` from `pt_head`, keeps a replay head with pseudolabeling, and inherits the foundation energy shifts. The replay dataset downloads only when fine-tuning is enabled. The short run demonstrates the workflow; it is not a converged model.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import warnings
import logging

warnings.filterwarnings("ignore")

class _MACEConversionWarningFilter(logging.Filter):
    def filter(self, record):
        return not record.getMessage().startswith("Default dtype ")

logging.getLogger().addFilter(_MACEConversionWarningFilter())
os.environ["PYTHONWARNINGS"] = "ignore"

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

PACKAGE_URL = "https://github.com/mdi-group/mace-field.git"
TUTORIAL_URL = "https://github.com/mdi-group/mace-field-tutorial.git"


def ensure_checkout(path, url):
    path = Path(path)
    if (path / ".git").is_dir():
        branch = subprocess.run(
            ["git", "-C", str(path), "branch", "--show-current"],
            capture_output=True,
            text=True,
            check=False,
        ).stdout.strip()
        if branch == "main":
            return
        subprocess.run(["git", "-C", str(path), "fetch", "origin", "main"], check=True)
        subprocess.run(
            ["git", "-C", str(path), "checkout", "-B", "main", "origin/main"],
            check=True,
        )
        return
    if path.exists() and any(path.iterdir()):
        raise RuntimeError(f"Refusing to clone over a non-empty directory: {path}")
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", "main", url, str(path)],
        check=True,
    )


if IN_COLAB:
    TUTORIAL_REPO = Path("/content/mace-field-tutorial")
    PACKAGE_REPO = Path("/content/mace-field")
    ensure_checkout(TUTORIAL_REPO, TUTORIAL_URL)
    ensure_checkout(PACKAGE_REPO, PACKAGE_URL)
else:
    CWD = Path.cwd().resolve()
    TUTORIAL_REPO = next(
        (
            path
            for path in (CWD, *CWD.parents)
            if (path / "tutorials/macefield/data").is_dir()
            and (path / "README.md").is_file()
        ),
        None,
    )
    if TUTORIAL_REPO is None:
        raise RuntimeError(
            "Run this notebook from a clone of mdi-group/mace-field-tutorial, "
            "or open it in Colab."
        )

    requested_repo = os.environ.get("MACEFIELD_REPO")
    if requested_repo:
        PACKAGE_REPO = Path(requested_repo).expanduser().resolve()
        if not (PACKAGE_REPO / "mace/modules/extensions.py").is_file():
            raise RuntimeError("MACEFIELD_REPO does not point to a MACE-Field checkout.")
        branch = subprocess.run(
            ["git", "-C", str(PACKAGE_REPO), "branch", "--show-current"],
            capture_output=True,
            text=True,
            check=False,
        ).stdout.strip()
        if branch != "main":
            raise RuntimeError("Set MACEFIELD_REPO to a checkout on the main branch.")
    else:
        candidates = [
            TUTORIAL_REPO.parent / "mace-field-main",
            TUTORIAL_REPO.parent / "mace-field",
        ]
        PACKAGE_REPO = None
        for candidate in candidates:
            if not (candidate / "mace/modules/extensions.py").is_file():
                continue
            branch = subprocess.run(
                ["git", "-C", str(candidate), "branch", "--show-current"],
                capture_output=True,
                text=True,
                check=False,
            ).stdout.strip()
            if branch == "main":
                PACKAGE_REPO = candidate.resolve()
                break
        if PACKAGE_REPO is None:
            PACKAGE_REPO = Path("/tmp/mace-field-main")
            ensure_checkout(PACKAGE_REPO, PACKAGE_URL)

os.environ["MACEFIELD_REPO"] = str(PACKAGE_REPO)
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"
sys.path.insert(0, str(PACKAGE_REPO))
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--editable", str(PACKAGE_REPO)],
    check=True,
)
os.chdir(TUTORIAL_REPO)

TUTORIAL_DIR = TUTORIAL_REPO / "tutorials/macefield"
DATA_DIR = TUTORIAL_DIR / "data"
print("Tutorial repository:", TUTORIAL_REPO)
print("MACE-Field source:", PACKAGE_REPO)


def run_streamed(command, *, cwd):
    """Stream a command's stdout and stderr into the notebook cell."""
    child_env = os.environ.copy()
    child_env["PYTHONUNBUFFERED"] = "1"
    child_env["PYTHONWARNINGS"] = "ignore"
    print("Running command:", command, flush=True)
    process = subprocess.Popen(
        command,
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=child_env,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="", flush=True)
    return_code = process.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)


## Download and verify the released model and replay data

The `MACEField-MH-1.model` checkpoint is fine-tuned from `mace-mh-1.model` on curated MP-Ferroelectric, MP-Dielectric, and Pheasy-MP-Phonon-v1.1 data. This notebook uses the checkpoint's `pt_head` for inference and replay. The release notes describe the training labels and validation limits. The model and replay assets are checked against their SHA-256 digests before use.


In [ ]:
import hashlib
import urllib.request

RELEASE_BASE = "https://github.com/mdi-group/mace-field/releases/download/1.1.0"
ASSET_DIR = TUTORIAL_DIR / "runs" / "foundation_assets"
ASSET_DIR.mkdir(parents=True, exist_ok=True)

ASSETS = {
    "MACEField-MH-1.model": "3e654f1cd043df8df68002f7d7eec626b3e9dce7763dedbbf3710188bff048d0",
    "MACEField-MH-1-replay-set.extxyz": "ecd678fff40a7e31033197bef52d39e9a2f52603532e2f5ae4e871aa368a1166",
}

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def download_verified(name, expected_sha256):
    path = ASSET_DIR / name
    if not path.is_file() or sha256_file(path) != expected_sha256:
        url = f"{RELEASE_BASE}/{name}"
        print("Downloading", name, "with progress updates...", flush=True)
        progress = {"bucket": -1}

        def report_progress(block_count, block_size, total_size):
            downloaded = block_count * block_size
            bucket = downloaded // (16 * 1024 * 1024)
            if bucket != progress["bucket"] or (total_size > 0 and downloaded >= total_size):
                if total_size > 0:
                    message = f"{downloaded / 1024**2:.1f}/{total_size / 1024**2:.1f} MiB"
                else:
                    message = f"{downloaded / 1024**2:.1f} MiB"
                print(f"  {name}: {message}", flush=True)
                progress["bucket"] = bucket

        urllib.request.urlretrieve(url, path, reporthook=report_progress)
    actual = sha256_file(path)
    if actual != expected_sha256:
        raise ValueError(f"Checksum mismatch for {name}: {actual}")
    print(name, "| SHA-256:", actual)
    return path

FOUNDATION_MODEL = download_verified(
    "MACEField-MH-1.model",
    ASSETS["MACEField-MH-1.model"],
)
REPLAY_ASSET_NAME = "MACEField-MH-1-replay-set.extxyz"


## Single-point analyses with the released `pt_head`

The following examples load the checksum-verified MACEField-MH-1 1.1.0 checkpoint. They use the bundled structures directly and do not depend on training outputs from another notebook. The Hessian is evaluated at zero electric field.


In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from ase.io import read
from ase.optimize import BFGS
from mace.calculators import MACECalculator

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_HEAD = "pt_head"
BTO_SAMPLE = DATA_DIR / "macefield_batio3_reference_valid.extxyz"
SIO2_SAMPLE = DATA_DIR / "sio2/SiO2-preprocessed.extxyz"


def clean_structure(path):
    atoms = read(path, index=0)
    atoms.info.clear()
    for key in list(atoms.arrays):
        if key not in {"numbers", "positions"}:
            atoms.arrays.pop(key)
    return atoms


def foundation_calculator(
    *,
    electric_field=(0.0, 0.0, 0.0),
    compute_polarization=True,
    compute_becs=True,
    compute_polarizability=True,
):
    return MACECalculator(
        model_paths=str(FOUNDATION_MODEL),
        model_type="MACEField",
        head=MODEL_HEAD,
        device=DEVICE,
        default_dtype="float32",
        electric_field=list(electric_field),
        compute_polarization=compute_polarization,
        compute_becs=compute_becs,
        compute_polarizability=compute_polarizability,
    )

print("MACEField model:", FOUNDATION_MODEL)
print("model head:", MODEL_HEAD, "| device:", DEVICE)


## BaTiO₃: polarization and BECs versus Ti displacement

Displace the Ti atom nearest the center of its oxygen cage along $z$. For each geometry, calculate the zero-field Ti $Z^*_{zz}$ and polarization at $E_z=-0.36$, 0, and +0.36 V/Å using the foundation `pt_head`. The final check compares the BEC with a small-displacement derivative of polarization at zero field.


In [ ]:
bto_source = clean_structure(BTO_SAMPLE)
bto_volume = bto_source.get_volume()
symbols = np.asarray(bto_source.get_chemical_symbols())
ti_candidates = np.flatnonzero(symbols == "Ti")
fractional = bto_source.get_scaled_positions(wrap=True)
center_delta = fractional[ti_candidates] - 0.5
center_delta -= np.rint(center_delta)
ti_index = int(ti_candidates[np.argmin(np.linalg.norm(center_delta, axis=1))])
print("BTO atoms:", len(bto_source), "| selected Ti index:", ti_index)
print("cell volume (Å³):", bto_volume)


def bto_at_displacement(displacement):
    atoms = bto_source.copy()
    atoms.positions[ti_index, 2] += float(displacement)
    atoms.info.clear()
    return atoms


displacements = np.linspace(-0.12, 0.12, 7)
field_values = (-0.36, 0.0, 0.36)
polarization_by_field = {field: [] for field in field_values}
ti_bec_zz = []
calc = foundation_calculator(compute_polarizability=False)

for disp_index, displacement in enumerate(displacements, start=1):
    atoms = bto_at_displacement(displacement)
    calc.electric_field = [0.0, 0.0, 0.0]
    calc.reset()
    atoms.calc = calc
    atoms.get_potential_energy()
    becs_now = np.asarray(calc.results["becs"], dtype=float).reshape(len(atoms), 3, 3)
    ti_bec_zz.append(float(becs_now[ti_index, 2, 2]))
    print(
        f"Ti displacement {disp_index}/{len(displacements)}: {displacement:+.3f} Å "
        f"| pt_head Z*zz={ti_bec_zz[-1]:+.4f} e",
        flush=True,
    )
    for field_z in field_values:
        calc.electric_field = [0.0, 0.0, float(field_z)]
        calc.reset()
        atoms.calc = calc
        atoms.get_potential_energy()
        pz = float(np.asarray(calc.results["polarization"])[2])
        polarization_by_field[field_z].append(pz)
        print(f"  Ez={field_z:+.3f} V/Å | Pz={pz:+.6e} e/Å²", flush=True)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for field_z in field_values:
    axes[0].plot(
        displacements,
        polarization_by_field[field_z],
        "o-",
        label=f"Ez={field_z:+.2f} V/Å",
    )
axes[0].set(
    xlabel="selected Ti displacement along z (Å)",
    ylabel="Pz (e/Å²)",
    title="MACEField-MH-1 polarization",
)
axes[0].legend()
axes[1].plot(displacements, ti_bec_zz, "o-", color="#9b4f96")
axes[1].set(
    xlabel="selected Ti displacement along z (Å)",
    ylabel="Ti Z*zz (e)",
    title="MACEField-MH-1 Born effective charge",
)
for ax in axes:
    ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()

# Local derivative check at the center of the displacement sweep.
d0 = 0.04
delta = 1.0e-3
calc.electric_field = [0.0, 0.0, 0.0]
calc.reset()
atoms0 = bto_at_displacement(d0)
atoms0.calc = calc
atoms0.get_potential_energy()
z_model = float(
    np.asarray(calc.results["becs"], dtype=float).reshape(len(atoms0), 3, 3)[ti_index, 2, 2]
)
pz_samples = []
for displacement in (d0 - delta, d0 + delta):
    atoms = bto_at_displacement(displacement)
    calc.reset()
    atoms.calc = calc
    atoms.get_potential_energy()
    pz_samples.append(float(np.asarray(calc.results["polarization"])[2]))
z_finite_difference = bto_volume * (pz_samples[1] - pz_samples[0]) / (2.0 * delta)
print("pt_head Ti Z*zz at d=0.04 Å (e):", z_model)
print("polarization finite-difference Z*zz at zero field (e):", z_finite_difference)


## α-quartz SiO₂: electronic and harmonic ionic dielectric response

Use the foundation `pt_head` to relax one bundled 72-atom frame at fixed cell, then evaluate polarization, electronic susceptibility, and BECs. Compute the zero-field Cartesian Hessian at that geometry and combine it with the BECs for a harmonic ionic contribution. The Hessian calculation is the costly step; the structure, model head, and cell are printed for provenance.


In [ ]:
sio2_atoms = clean_structure(SIO2_SAMPLE)
print("SiO2 atoms:", len(sio2_atoms), "| initial volume (Å³):", sio2_atoms.get_volume())
relax_calc = foundation_calculator(
    compute_polarization=False,
    compute_becs=False,
    compute_polarizability=False,
)
sio2_atoms.calc = relax_calc
print("Fixed-cell BFGS relaxation with MACEField-MH-1 pt_head:", flush=True)
BFGS(sio2_atoms, logfile="-").run(fmax=0.05, steps=60)
print("energy after relaxation (eV):", float(sio2_atoms.get_potential_energy()))
print(
    "maximum force after relaxation (eV/Å):",
    float(np.linalg.norm(sio2_atoms.get_forces(), axis=1).max()),
)
print("relaxed volume (Å³):", sio2_atoms.get_volume())


### Electronic response and Born effective charges

The calculator's `polarizability` result is the volume-normalized electronic susceptibility used here, so $\epsilon_\infty=I+\chi_\mathrm{electronic}$. The BEC tensor has units of elementary charge.


In [ ]:
response_calc = foundation_calculator()
sio2_atoms.calc = response_calc
sio2_atoms.get_potential_energy()
polarization = np.asarray(response_calc.results["polarization"], dtype=float)
becs = np.asarray(response_calc.results["becs"], dtype=float).reshape(len(sio2_atoms), 3, 3)
chi_electronic = np.asarray(response_calc.results["polarizability"], dtype=float).reshape(3, 3)
epsilon_infinity = np.eye(3) + chi_electronic

print("pt_head polarization (e/Å²):", polarization)
print("BEC tensor shape:", becs.shape)
print("mean BEC over atoms (e):")
print(becs.mean(axis=0))
print("electronic susceptibility:")
print(chi_electronic)
print("electronic dielectric tensor epsilon_infinity:")
print(epsilon_infinity)


### Hessian and harmonic ionic dielectric contribution

`get_hessian` differentiates the zero-field model energy. The ionic estimate uses the positive Hessian modes above the acoustic cutoff; any additional negative modes signal that this harmonic estimate is not a stable-material dielectric constant.


In [ ]:
print(
    f"Calculating the dense {3 * len(sio2_atoms)} x {3 * len(sio2_atoms)} "
    "zero-field Cartesian Hessian with pt_head...",
    flush=True,
)
hessian = np.asarray(response_calc.get_hessian(sio2_atoms)).reshape(3 * len(sio2_atoms), -1)
hessian = 0.5 * (hessian + hessian.T)
print("Hessian shape:", hessian.shape, "| units: eV/Å²")
print("maximum asymmetry after symmetrization:", float(np.max(np.abs(hessian - hessian.T))))

eigenvalues, eigenvectors = np.linalg.eigh(hessian)
cutoff = max(float(np.max(np.abs(eigenvalues))) * 1.0e-8, 1.0e-8)
stable_modes = eigenvalues > cutoff
negative_modes = int(np.count_nonzero(eigenvalues < -cutoff))
hessian_pinv = (
    eigenvectors[:, stable_modes] / eigenvalues[stable_modes]
) @ eigenvectors[:, stable_modes].T

# Z[polarization direction, Cartesian displacement coordinate]
z_matrix = np.transpose(becs, (1, 0, 2)).reshape(3, -1)
eps0_e_per_v_angstrom = 8.8541878128e-12 / 1.602176634e-19 / 1.0e10
chi_ionic = (
    z_matrix @ hessian_pinv @ z_matrix.T
    / (sio2_atoms.get_volume() * eps0_e_per_v_angstrom)
)
epsilon_static_harmonic = epsilon_infinity + chi_ionic

print("Hessian eigenvalues below the acoustic cutoff:", negative_modes)
print("retained positive modes:", int(np.count_nonzero(stable_modes)))
print("ionic susceptibility:")
print(chi_ionic)
print("harmonic static dielectric tensor:")
print(epsilon_static_harmonic)
if negative_modes:
    print(
        "Warning: this pt_head-relaxed structure has unstable modes; "
        "do not interpret the harmonic ionic estimate as a stable dielectric constant."
    )


## Optional: fine-tune a BTO head from MACEField-MH-1

When enabled, the next cell downloads the release replay set, selects replay structures, and writes a multi-head configuration with `pt_head` replay and a new `ft_head` trained on the 48-frame BTO training split. The 11-frame validation split stays separate. Replay labels are generated from the MACEField-MH-1 checkpoint.

The cell streams replay selection and training logs as they arrive. Each epoch prints validation loss and the RMSEs for available mechanical and response labels. Set `MACEFIELD_RUN_FINETUNE=1` before running the cell to launch the one-epoch demonstration. Increase replay samples, epochs, and patience only after checking GPU memory and reviewing the split. The release also provides a fine-tuning script.


In [ ]:
import os
import subprocess
import sys
import time
import yaml

BTO_TRAIN = DATA_DIR / "macefield_batio3_reference_train.extxyz"
BTO_VALID = DATA_DIR / "macefield_batio3_reference_valid.extxyz"
RUN_DIR = TUTORIAL_DIR / "runs" / f"foundation_batio3_{time.strftime('%Y%m%d_%H%M%S')}"
RUN_DIR.mkdir(parents=True, exist_ok=False)
SELECTED_REPLAY = RUN_DIR / "selected_replay.xyz"
CONFIG = RUN_DIR / "finetune.yml"

RUN_FINETUNE = os.environ.get("MACEFIELD_RUN_FINETUNE", "0") == "1"
REPLAY_DATA = ASSET_DIR / REPLAY_ASSET_NAME
if RUN_FINETUNE:
    REPLAY_DATA = download_verified(REPLAY_ASSET_NAME, ASSETS[REPLAY_ASSET_NAME])

NUM_REPLAY = int(os.environ.get("MACEFIELD_TUTORIAL_REPLAY_SAMPLES", "32"))
EPOCHS = int(os.environ.get("MACEFIELD_TUTORIAL_EPOCHS", "1"))
BATCH_SIZE = int(os.environ.get("MACEFIELD_TUTORIAL_BATCH_SIZE", "1"))

selector_command = [
    sys.executable, "-m", "mace.cli.fine_tuning_select",
    "--configs_pt", str(REPLAY_DATA),
    "--configs_ft", str(BTO_TRAIN),
    "--num_samples", str(NUM_REPLAY),
    "--subselect", "random",
    "--model", str(FOUNDATION_MODEL),
    "--model_type", "MACEField",
    "--output", str(SELECTED_REPLAY),
    "--head_pt", "pt_head",
    "--head_ft", "ft_head",
    "--filtering_type", "combinations",
    "--weight_pt", "1.0",
    "--weight_ft", "1.0",
    "--device", DEVICE,
    "--default_dtype", "float32",
    "--seed", "23",
]

finetune_config = {
    "name": "MACEField-MH-1-BaTiO3-tutorial",
    "model": "MACEField",
    "loss": "universal_field",
    "foundation_model": str(FOUNDATION_MODEL),
    "foundation_head": "pt_head",
    "multiheads_finetuning": True,
    "pseudolabel_replay": True,
    "pt_train_file": str(SELECTED_REPLAY),
    "heads": {
        "ft_head": {
            "train_file": str(BTO_TRAIN),
            "valid_file": str(BTO_VALID),
        }
    },
    "valid_fraction": 0.2,
    "E0s": "foundation",
    "energy_weight": 1.0,
    "forces_weight": 10.0,
    "stress_weight": 1.0,
    "polarization_weight": 10.0,
    "becs_weight": 100.0,
    "polarizability_weight": 100.0,
    "compute_forces": True,
    "compute_stress": True,
    "compute_polarization": True,
    "compute_becs": True,
    "compute_polarizability": True,
    "batch_size": BATCH_SIZE,
    "valid_batch_size": 1,
    "num_workers": 0,
    "max_num_epochs": EPOCHS,
    "patience": max(3, EPOCHS),
    "eval_interval": 1,
    "error_table": "PerAtomFieldRMSE",
    "default_dtype": "float32",
    "device": DEVICE,
    "seed": 23,
    "model_dir": str(RUN_DIR / "models"),
    "checkpoints_dir": str(RUN_DIR / "checkpoints"),
    "log_dir": str(RUN_DIR / "logs"),
    "results_dir": str(RUN_DIR / "results"),
    "work_dir": str(RUN_DIR / "work"),
}
for output_key in ("model_dir", "checkpoints_dir", "log_dir", "results_dir", "work_dir"):
    Path(finetune_config[output_key]).mkdir(parents=True, exist_ok=True)
CONFIG.write_text(yaml.safe_dump(finetune_config, sort_keys=False))
print("Fine-tuning configuration:", CONFIG)
print("Selected replay output:", SELECTED_REPLAY)
print("Set MACEFIELD_RUN_FINETUNE=1 to run with", NUM_REPLAY, "replay frames and", EPOCHS, "epoch(s).")
print("When enabled, selector and training output stream into this cell; each epoch reports validation loss and response RMSEs.", flush=True)

if RUN_FINETUNE:
    run_streamed(selector_command, cwd=PACKAGE_REPO)
    run_streamed(
        [sys.executable, "-m", "mace.cli.run_train", "--config", str(CONFIG)],
        cwd=PACKAGE_REPO,
    )
    print("Fine-tuned models:", list((RUN_DIR / "models").glob("*.model")))
else:
    print("Training skipped. Foundation inference above ran independently.")


## Interpretation

The `MACEField-MH-1.model` checkpoint and its replay set are separate, pinned inputs. Fine-tuning keeps the foundation replay head while adding a target head for the BTO labels. The one-epoch example only demonstrates the workflow; it is not a converged transfer-learning result.
